# House Prices: Advanced Regression Techniques

Stacked ensemble of 7 Optuna-tuned regressors for the Kaggle *House Prices* competition.
Team project (4 students), Data Science course, KMUTT.

**Result:** rank #419 (top 8%), RMSE 0.12119 on the leaderboard, up from #734 (RMSE 0.12603) with our best single model (tuned XGBoost) in the previous phase.

> Outputs were cleared when this notebook was tidied for GitHub (comments and unused code removed; the modelling logic is unchanged from the submitted version). The results below were recorded from the original Kaggle run in November 2025.

| Model (5-fold CV RMSE on log price, best of 50 Optuna trials) | RMSE |
|---|---|
| XGBoost | 0.1172 |
| KernelRidge | 0.1211 |
| LightGBM | 0.1255 |
| ExtraTrees | 0.1304 |
| ElasticNet | 0.1338 |
| Lasso | 0.1344 |
| Ridge | 0.1369 |
| **Stacked ensemble (Kaggle leaderboard)** | **0.12119** |

In [ ]:
import numpy as np
import pandas as pd
import warnings

warnings.filterwarnings("ignore")

## Data loading

In [ ]:
df = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
test = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv")

## Feature engineering

New features for size, age, amenities and quality, wrapped in a `FunctionTransformer` so they run inside the scikit-learn pipeline.

In [ ]:
from sklearn.preprocessing import FunctionTransformer

drop_cols = ["Id", "Alley", "PoolQC", "Fence", "MiscFeature"]


def added_features(df):
    df_out = df.copy()

    # Intended to drop high-null columns. The result is not assigned,
    # so nothing is dropped (see "Known issues" below).
    df_out.drop(columns=drop_cols, errors="ignore")

    # Age
    df_out["PropertyAge"] = df_out["YrSold"] - df_out["YearBuilt"]

    # Size
    df_out["TotalSF"] = df_out["TotalBsmtSF"] + df_out["1stFlrSF"] + df_out["2ndFlrSF"]
    df_out["TotalBath"] = df_out["FullBath"] + 0.5 * df_out["HalfBath"] + df_out["BsmtFullBath"] + 0.5 * df_out["BsmtHalfBath"]

    # Remodelled since built
    df_out["HasRemodeled"] = (df_out["YearRemodAdd"] != df_out["YearBuilt"]).astype(object)

    # Amenity flags
    df_out["HasPool"] = (df_out["PoolArea"] > 0).astype(int)
    df_out["HasBasement"] = (df_out["TotalBsmtSF"] > 0).astype(int)
    df_out["HasFireplace"] = (df_out["Fireplaces"] > 0).astype(int)
    df_out["Has2ndFloor"] = (df_out["2ndFlrSF"] > 0).astype(object)
    df_out["HasGarage"] = (df_out["GarageArea"] > 0).astype(object)

    # Numeric columns also used as categories
    df_out["YrSold_cat"] = df_out["YrSold"].astype(object)
    df_out["MoSold_cat"] = df_out["MoSold"].astype(object)
    df_out["YearBuilt_cat"] = df_out["YearBuilt"].astype(object)
    df_out["MSSubClass_cat"] = df_out["MSSubClass"].astype(object)

    # Quality interactions
    df_out["OverallGrade"] = df_out["OverallQual"] * df_out["OverallCond"]
    df_out["QualityArea"] = df_out["OverallQual"] * df_out["GrLivArea"]

    return df_out


feature_transformer = FunctionTransformer(added_features)

## Preprocessing pipeline

Categorical columns: missing values become `"None"` (usually meaning the home has no such feature), then one-hot encoding.
Numerical columns: mean imputation, then standardisation. The target is modelled as `log(SalePrice)`.

In [ ]:
new_cols_categorical = pd.Index(["HasRemodeled", "Has2ndFloor", "HasGarage",
                                 "YrSold_cat", "MoSold_cat", "YearBuilt_cat",
                                 "MSSubClass_cat"])
new_cols_numeric = pd.Index(["PropertyAge", "TotalSF", "TotalBath",
                             "HasFireplace", "HasBasement", "HasPool",
                             "OverallGrade", "QualityArea"])

categorical_columns = df.select_dtypes(include=["object", "category"]).columns.append(new_cols_categorical)
numerical_columns = df.select_dtypes(include=["int64", "float64"]).columns.append(new_cols_numeric)

# The target is not a feature
numerical_columns = numerical_columns.drop("SalePrice")

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

cat_transform = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value="None")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse=False)),
])

num_transform = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler()),
])

In [ ]:
preprocessor = ColumnTransformer(transformers=[("cat", cat_transform, categorical_columns),
                                               ("num", num_transform, numerical_columns)],
                                 remainder="passthrough")

pipeline = Pipeline(steps=[
    ("engineer", feature_transformer),
    ("preprocessor", preprocessor),
])

X = df.drop("SalePrice", axis=1)
y = np.log(df["SalePrice"])  # prices are right-skewed

X_preprocessed = pipeline.fit_transform(X)
test_preprocessed = pipeline.transform(test)

## Hyperparameter tuning (Optuna)

Each model is tuned for 50 trials, scored by 5-fold cross-validated RMSE on log price.
CatBoost was also tried, but its tuning was stopped before it finished, so it is not included here or in the final stack.

In [ ]:
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import Ridge, ElasticNet, Lasso
from sklearn.kernel_ridge import KernelRidge
from sklearn.ensemble import ExtraTreesRegressor
import optuna

In [ ]:
objectives = {}


def objective_lgbm(trial):
    params = {
        "n_estimators": 2000,
        "verbose": -1,
        "random_state": 42,
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        # Tree structure
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "num_leaves": trial.suggest_int("num_leaves", 20, 100),
        # Regularisation
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 100),
        "min_split_gain": trial.suggest_float("min_split_gain", 0.0, 0.5),
        # Sampling (subsample_freq must be > 0 for subsample to apply)
        "subsample": trial.suggest_float("subsample", 0.5, 0.95),
        "subsample_freq": trial.suggest_int("subsample_freq", 1, 5),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 0.95),
        # L1 / L2
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-5, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-5, 10.0, log=True),
    }
    model = LGBMRegressor(**params)
    score = -cross_val_score(model, X_preprocessed, y,
                             cv=5, scoring="neg_root_mean_squared_error", n_jobs=-1).mean()
    return score

objectives["LightGBM"] = objective_lgbm


def objective_ridge(trial):
    alpha = trial.suggest_float("alpha", 1e-5, 20000.0, log=True)
    tol = trial.suggest_float("tol", 1e-5, 1e-3, log=True)
    model = Ridge(alpha=alpha, tol=tol, random_state=42)
    score = -cross_val_score(model, X_preprocessed, y,
                             cv=5, scoring="neg_root_mean_squared_error").mean()
    return score

objectives["Ridge"] = objective_ridge


def objective_elastic(trial):
    alpha = trial.suggest_float("alpha", 1e-5, 100.0, log=True)
    # 0 = Ridge, 1 = Lasso; start at 0.01 because ElasticNet should not be used with exactly 0
    l1_ratio = trial.suggest_float("l1_ratio", 0.01, 1.0)
    tol = trial.suggest_float("tol", 1e-5, 1e-2, log=True)
    selection = trial.suggest_categorical("selection", ["cyclic", "random"])
    model = ElasticNet(
        alpha=alpha,
        l1_ratio=l1_ratio,
        tol=tol,
        selection=selection,
        random_state=42,
        max_iter=10000,  # small alphas need more iterations to converge
    )
    score = -cross_val_score(model, X_preprocessed, y,
                             cv=5, scoring="neg_root_mean_squared_error").mean()
    return score

objectives["ElasticNet"] = objective_elastic


def objective_krr(trial):
    alpha = trial.suggest_float("alpha", 1e-4, 10.0, log=True)
    # No linear kernel: it would duplicate Ridge, which is already in the stack
    kernel = trial.suggest_categorical("kernel", ["polynomial", "rbf", "laplacian"])
    gamma = trial.suggest_float("gamma", 1e-4, 5.0, log=True)
    if kernel == "polynomial":
        degree = trial.suggest_int("degree", 2, 3)
        coef0 = trial.suggest_float("coef0", 0.1, 10.0)
        model = KernelRidge(alpha=alpha, kernel=kernel, gamma=gamma, degree=degree, coef0=coef0)
    else:
        model = KernelRidge(alpha=alpha, kernel=kernel, gamma=gamma)
    score = -cross_val_score(model, X_preprocessed, y,
                             cv=5, scoring="neg_root_mean_squared_error").mean()
    return score

objectives["KernelRidge"] = objective_krr


def objective_xgb(trial):
    params = {
        "objective": "reg:squarederror",
        "tree_method": "hist",
        "random_state": 42,
        "n_estimators": 2000,
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        # Conservative splitting
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 20),
        "gamma": trial.suggest_float("gamma", 1e-4, 1.0, log=True),
        # Row / column sampling
        "subsample": trial.suggest_float("subsample", 0.5, 0.95),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 0.95),
        # L1 / L2
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
    }
    model = XGBRegressor(**params)
    score = -cross_val_score(model, X_preprocessed, y,
                             cv=5, scoring="neg_root_mean_squared_error").mean()
    return score

objectives["XGBoost"] = objective_xgb


def objective_lasso(trial):
    alpha = trial.suggest_float("alpha", 1e-4, 10.0, log=True)
    model = Lasso(alpha=alpha, random_state=42, max_iter=10000)
    score = -cross_val_score(model, X_preprocessed, y,
                             cv=5, scoring="neg_root_mean_squared_error").mean()
    return score

objectives["Lasso"] = objective_lasso


def objective_et(trial):
    params = {
        "n_estimators": 1000,
        "n_jobs": -1,
        "random_state": 42,
        "max_depth": trial.suggest_int("max_depth", 3, 25),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 15),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 10),
        "max_features": trial.suggest_float("max_features", 0.5, 1.0),
        "bootstrap": trial.suggest_categorical("bootstrap", [True, False]),
    }
    model = ExtraTreesRegressor(**params)
    score = -cross_val_score(model, X_preprocessed, y,
                             cv=5, scoring="neg_root_mean_squared_error").mean()
    return score

objectives["ExtraTrees"] = objective_et

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)
results = {}            # best parameters per model
model_performance = {}  # best CV RMSE per model

print(f"Tuning {len(objectives)} models...")

for model_name, objective_func in objectives.items():
    print(f"Optimising {model_name}...")
    study = optuna.create_study(direction="minimize")
    study.optimize(objective_func, n_trials=50, show_progress_bar=True)

    results[model_name] = study.best_params
    model_performance[model_name] = study.best_value

    print(f"{model_name} best CV RMSE: {study.best_value:.5f}")

leaderboard = pd.DataFrame(list(model_performance.items()), columns=["Model", "RMSE"])
print(leaderboard.sort_values("RMSE"))

## Stacked ensemble

The seven tuned models are combined by `StackingRegressor`, with `RidgeCV` as the meta-model trained on 5-fold out-of-fold predictions.

In [ ]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import RidgeCV

lgbm_final = LGBMRegressor(
    n_estimators=1000,
    random_state=42,
    verbose=-1,
    **results["LightGBM"]
)

xgb_final = XGBRegressor(
    n_estimators=1000,
    objective="reg:squarederror",
    tree_method="hist",
    random_state=42,
    **results["XGBoost"]
)

ridge_final = Ridge(random_state=42, **results["Ridge"])

elastic_final = ElasticNet(random_state=42, max_iter=20000, **results["ElasticNet"])

lasso_final = Lasso(random_state=42, max_iter=10000, **results["Lasso"])

krr_final = KernelRidge(**results["KernelRidge"])

et_final = ExtraTreesRegressor(
    n_estimators=1000,
    n_jobs=-1,
    random_state=42,
    **results["ExtraTrees"]
)

estimators = [
    ("lgbm", lgbm_final),
    ("xgb", xgb_final),
    ("ridge", ridge_final),
    ("elastic", elastic_final),
    ("lasso", lasso_final),
    ("krr", krr_final),
    ("et", et_final),
]

stacking_model = StackingRegressor(
    estimators=estimators,
    final_estimator=RidgeCV(),
    cv=5,
    n_jobs=-1,
    passthrough=False,
)

In [ ]:
stacking_model.fit(X_preprocessed, y)

## How similar are the base models?

Stacking helps most when base models make different errors. This plots the correlation of their out-of-fold predictions.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_predict

prediction_df = pd.DataFrame()

for name, model in estimators:
    prediction_df[name] = cross_val_predict(model, X_preprocessed, y, cv=5, n_jobs=-1)

corr_matrix = prediction_df.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

plt.figure(figsize=(12, 10))
sns.set_style("white")
sns.heatmap(corr_matrix, annot=True, fmt=".3f", cmap="coolwarm", mask=mask,
            vmin=0.8, vmax=1.0, linewidths=0.5, square=True, cbar_kws={"shrink": .75})
plt.title("Base model prediction correlation", fontsize=16)
plt.show()

## Submission

In [ ]:
final_predictions = stacking_model.predict(test_preprocessed)
final_pred = np.expm1(final_predictions)  # see "Known issues": np.exp matches np.log

submission = pd.DataFrame({
    "Id": test["Id"],
    "SalePrice": final_pred,
})

submission_filename = "submission_stacking_v1.csv"
submission.to_csv(submission_filename, index=False)
print(f"Saved {submission_filename}")
print(submission.head())

## Known issues

Found when reviewing the notebook after the competition. They are left unchanged above so the notebook matches the submitted result.

1. **Log transform mismatch.** The target is `np.log(SalePrice)`, but predictions are converted back with `np.expm1` (the inverse of `log1p`), so every prediction is $1 too low. Negligible at these prices, but the fix is `np.exp`, or `log1p` with `expm1`.
2. **Columns not dropped.** `df_out.drop(...)` is not assigned back, so the high-null columns stay in and are imputed and encoded instead. Fix: `df_out = df_out.drop(...)`.
3. **Tree count differs between tuning and the final models.** LightGBM and XGBoost were tuned with 2,000 trees but rebuilt with 1,000 for stacking. With the low learning rates found (for example 0.008 for XGBoost), the final models are probably under-trained. Fix: use the same `n_estimators`, or tune it.
4. **Slightly optimistic CV scores.** The pipeline is fitted on all training data before cross-validation, so imputation means and scaling see the validation folds. Fix: put the pipeline inside the cross-validated estimator.
5. **Not fully reproducible.** Optuna has no fixed seed. Fix: `optuna.create_study(sampler=optuna.samplers.TPESampler(seed=42))`.